# Text Encoding

Text encoding is used to convert categorical text data into numbers so machine learning models can understand it.

In this notebook, we will learn three common scikit-learn methods: ordinal encoding, label encoding, and one-hot encoding.

## Ordinal Encoding

It is used when the categories have a meaningful order. Example: `Low`, `Medium`, and `High` can be encoded as `0`, `1`, and `2`.

This works because the categories already have a ranking. It is mainly used for input features where the order matters.

In [10]:
# Import pandas and scikit-learn encoders
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

In [11]:
# Create a sample data frame for ordinal encoding
data_ordinal = pd.DataFrame({
    'size': ['Small', 'Medium', 'Large', 'Medium', 'Small']
})

# Show the original data
data_ordinal

,size
0,Small
1,Medium
2,Large
3,Medium
4,Small


In [12]:
# Create and apply an ordinal encoder
ordinal_encoder = OrdinalEncoder(categories=[['Small', 'Medium', 'Large']])
data_ordinal['size_encoded'] = ordinal_encoder.fit_transform(data_ordinal[['size']]).astype(int)

# Show the encoded data
data_ordinal

,size,size_encoded
0,Small,0
1,Medium,1
2,Large,2
3,Medium,1
4,Small,0


## Label Encoding

Label encoding is mainly used for output or target variables. It gives each class label a unique number, such as `No = 0` and `Yes = 1`.

According to the scikit-learn documentation, `LabelEncoder` should be used for target values `y`, not for input features `X`.

In [13]:
# Create a sample target variable for label encoding
target = pd.Series(['Yes', 'No', 'Yes', 'No', 'Yes'], name='purchased')

# Show the original target values
target

0    Yes
1     No
2    Yes
3     No
4    Yes
Name: purchased, dtype: str

In [14]:
# Create and apply a label encoder
label_encoder = LabelEncoder()
target_encoded = pd.Series(
    label_encoder.fit_transform(target),
    name='purchased_encoded'
 )

# Show the encoded target values
target_encoded

0    1
1    0
2    1
3    0
4    1
Name: purchased_encoded, dtype: int64

## One-Hot Encoding

One-hot encoding is used when categories do not have a meaningful order. We use this because not every category in a dataset has some ranking or level.

For example, colors like `Red`, `Blue`, and `Green` do not have a natural order. One-hot encoding creates separate columns for each category and marks the matching category with `1` and the others with `0`.

### Why do we sometimes drop one column?

We often drop one column in one-hot encoding to avoid redundant information.

If a feature has $k$ categories, one-hot encoding creates $k$ binary columns. But once you know the values of $k - 1$ columns, the last one is automatically determined. For example, if the categories are `Red`, `Blue`, and `Green`:

- `Red = 0`
- `Blue = 0`

then `Green` must be `1`.

That means the columns are perfectly dependent on each other. In linear models like linear regression or logistic regression, this causes multicollinearity, often called the dummy variable trap. Dropping one column removes that redundancy without losing information.

Example:

- Full encoding:
  - `Red` → `[1, 0, 0]`
  - `Blue` → `[0, 1, 0]`
  - `Green` → `[0, 0, 1]`

- Drop-first encoding:
  - `Red` → `[0, 0]`
  - `Blue` → `[1, 0]`
  - `Green` → `[0, 1]`

Here, the missing case represents the dropped base category.

A few practical rules:

- Drop one column when using linear or generalized linear models.
- It is often not necessary for tree-based models like decision trees, random forests, and XGBoost.
- In scikit-learn, you do this with `OneHotEncoder(drop='first')`.

If you want, I can update your one-hot encoding cell in the notebook to use `drop='first'` and add a short explanation there.

In [15]:
# Create a sample data frame for one-hot encoding
data_onehot = pd.DataFrame({
    'color': ['Red', 'Blue', 'Green', 'Blue', 'Red']
})

# Show the original data
data_onehot

,color
0,Red
1,Blue
2,Green
3,Blue
4,Red


In [29]:
# Create one-hot encoded data without dropping any column
onehot_encoder_full = OneHotEncoder(sparse_output=False)
encoded_full_array = onehot_encoder_full.fit_transform(data_onehot[['color']])

onehot_encoded_full = pd.DataFrame(
    encoded_full_array,
    columns=onehot_encoder_full.get_feature_names_out(['color']),
    index=data_onehot.index
).astype(int)

print('Without dropping a column:', onehot_encoded_full.shape)

Without dropping a column: (5, 3)


In [24]:
# One-hot encoded data without dropping any column
onehot_encoded_full

,color_Blue,color_Green,color_Red
0,0,0,1
1,1,0,0
2,0,1,0
3,1,0,0
4,0,0,1


In [31]:
# Print the encoding for Blue without dropping any column
blue_sample = pd.DataFrame({'color': ['Blue']})
blue_full = onehot_encoder_full.transform(blue_sample).astype(int)

print('Blue without dropping a column:', blue_full.tolist()[0])

Blue without dropping a column: [1, 0, 0]


In [30]:
# Create one-hot encoded data after dropping the first column
onehot_encoder_drop = OneHotEncoder(drop='first', sparse_output=False)
encoded_drop_array = onehot_encoder_drop.fit_transform(data_onehot[['color']])

onehot_encoded_drop = pd.DataFrame(
    encoded_drop_array,
    columns=onehot_encoder_drop.get_feature_names_out(['color']),
    index=data_onehot.index
).astype(int)

# Keep the original variable name for later cells
onehot_encoded = onehot_encoded_full

print('With drop="first":', onehot_encoded_drop.shape)

With drop="first": (5, 2)


In [33]:
# Print the encoding for Blue with drop='first'
blue_sample = pd.DataFrame({'color': ['Blue']})
blue_drop = onehot_encoder_drop.transform(blue_sample).astype(int)

print('Blue with drop="first":', blue_drop.tolist()[0])

Blue with drop="first": [0, 0]
